In [ ]:
# --- Cell 1: Import Libraries ---
import os
import torch
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn as nn
import random
import torchvision.transforms.functional as TF


import segmentation_models_pytorch as smp

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

PyTorch Version: 2.5.1
Using device: cuda


In [ ]:
# --- Cell 2 : Dataset ที่รองรับ ImageNet Transform ---
class GlassMaskDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_transform=None, mask_transform=None, augment=False):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_transform = img_transform   # สำหรับรูป (มี Normalize)
        self.mask_transform = mask_transform # สำหรับ Mask (แค่ Resize)
        self.augment = augment
        self.image_ids = list(range(30000))

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        image = Image.open(img_path).convert("RGB")
        
        sub_folder = str(img_id // 2000)
        mask_path = os.path.join(self.mask_dir, sub_folder, f"{str(img_id).zfill(5)}_eye_g.png")
        
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
        else:
            mask = Image.fromarray(np.zeros((512, 512), dtype=np.uint8))

        # Augmentation
        if self.augment:
            if random.random() > 0.5:
                angle = random.randint(-30, 30)
                image = TF.rotate(image, angle)
                mask = TF.rotate(mask, angle)
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

        # Apply Transforms
        if self.img_transform:
            image = self.img_transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)
            
        # Mask Thresholding
        mask = (mask > 0).float()
            
        return image, mask

In [ ]:
# --- Cell 3: Prepare Data & Normalization ---

IMG_DIR = r"D:\senior_project\dataset\CelebAMask-HQ\CelebA-HQ-img"
MASK_DIR = r"D:\senior_project\dataset\CelebAMask-HQ\CelebAMask-HQ-mask-anno"

BATCH_SIZE = 16 
EPOCHS = 20  

# 1. Transform สำหรับ "รูปภาพ" (Normalize ตามสูตร ImageNet)
img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Transform สำหรับ "Mask"
mask_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# 3. สร้าง Dataset
train_ds_full = GlassMaskDataset(IMG_DIR, MASK_DIR, img_transform, mask_transform, augment=True)
val_ds_full = GlassMaskDataset(IMG_DIR, MASK_DIR, img_transform, mask_transform, augment=False)

# 4. Split Data
indices = torch.randperm(len(train_ds_full)).tolist()
n_val = int(len(train_ds_full) * 0.1)
train_indices, val_indices = indices[n_val:], indices[:n_val]

train_set = torch.utils.data.Subset(train_ds_full, train_indices)
val_set = torch.utils.data.Subset(val_ds_full, val_indices)

# 5. DataLoader
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train Size: {len(train_set)} | Val Size: {len(val_set)}")

Train Size: 27000 | Val Size: 3000


In [ ]:
# --- Cell 4: Define Model (Transfer Learning) ---

# สร้าง U-Net
model = smp.Unet(
    encoder_name="resnet34",        
    encoder_weights="imagenet",    
    in_channels=3,                 
    classes=1,                      # Output: Mask ขาวดำ
)

model.to(device)
print("✅ โหลดโมเดล U-Net (ResNet34 Backbone) เรียบร้อย!")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

c:\Users\Asus\anaconda3\envs\gan_project\lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--smp-hub--resnet34.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

✅ โหลดโมเดล U-Net (ResNet34 Backbone) เรียบร้อย!


In [ ]:
# --- Cell 5: Training Loop ---
import time

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def train_model(model, epochs):
    best_val_loss = float('inf')
    
    print(f"--- เริ่มเทรน {epochs} Epochs ---")
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        # Train
        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(train_loader)
        
        # Val
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()
        
        val_epoch_loss = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}/{epochs} | Train: {epoch_loss:.5f} | Val: {val_epoch_loss:.5f}", end='')
        
        # Save Best
        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            torch.save(model.state_dict(), "best_unet_resnet34.pth") 
            print(" --> 🏆 New Best Model!")
        else:
            print("")

train_model(model, EPOCHS)

--- เริ่มเทรนโมเดลเทพ (ResNet34) 20 Epochs ---
Epoch 1/20 | Train: 0.05023 | Val: 0.00767 --> 🏆 New Best Model!
Epoch 2/20 | Train: 0.00474 | Val: 0.00308 --> 🏆 New Best Model!
Epoch 3/20 | Train: 0.00257 | Val: 0.00190 --> 🏆 New Best Model!
Epoch 4/20 | Train: 0.00205 | Val: 0.00203
Epoch 5/20 | Train: 0.00166 | Val: 0.00135 --> 🏆 New Best Model!
Epoch 6/20 | Train: 0.00167 | Val: 0.00147
Epoch 7/20 | Train: 0.00141 | Val: 0.00154
Epoch 8/20 | Train: 0.00138 | Val: 0.00177
Epoch 9/20 | Train: 0.00151 | Val: 0.00157
Epoch 10/20 | Train: 0.00124 | Val: 0.00209
Epoch 11/20 | Train: 0.00130 | Val: 0.00157
Epoch 12/20 | Train: 0.00125 | Val: 0.00169
Epoch 13/20 | Train: 0.00132 | Val: 0.00146
Epoch 14/20 | Train: 0.00118 | Val: 0.00140
Epoch 15/20 | Train: 0.00109 | Val: 0.00148
Epoch 16/20 | Train: 0.00117 | Val: 0.00165
Epoch 17/20 | Train: 0.00109 | Val: 0.00134 --> 🏆 New Best Model!
Epoch 18/20 | Train: 0.00105 | Val: 0.00182
Epoch 19/20 | Train: 0.00103 | Val: 0.00127 --> 🏆 New Best M